# 00. 데이터 이해와 품질 점검

이 단계에서 발견한 시간 오류·주간 반복·비생산 시 전력·월별 변화가 01 전처리와 이후 두 모델의 입력 및 자원최적화 제약을 설계하는 근거입니다. 1주 전 전력의 상관은 약 0.744이고 24시간 전은 약 0.397이지만, 상관이 곧 예측 성능이나 인과효과는 아닙니다.

| 항목 | 내용 |
|---|---|
| 목적 | 원본 구조, 시간 품질, 생산·전력 관계와 피크 패턴 확인 |
| 입력 | 원본 제조 전력 CSV |
| 출력 | `results/eda/` 중심의 표와 그림 |
| 선택 데이터 | 모델 선택에 사용하지 않는 탐색 단계 |
| 다음 단계 | `01_preprocessing.ipynb` |

---

# 00. 제조 전력 데이터 EDA
시간·생산·기상 조건에 따른 전력 패턴을 시각적으로 확인합니다. 위에서부터 순서대로 실행하세요.

- 그래프는 셀 출력으로만 표시합니다.
- 원본 CSV는 수정하지 않습니다. 분석용 CSV와 요약표는 `results/eda/csv`에 저장합니다.
- 전력의 물리적 단위와 측정 방식은 확정하지 않고 **전력값**으로 표시합니다.
- 통계적 피크 기준은 탐색용입니다. 모델 개발 시에는 학습 데이터에서만 다시 계산해야 합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'dataset' / 'okm_augumented_2021.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'dataset' / 'okm_augumented_2021.csv'
CSV_DIR = ROOT / 'results' / 'eda' / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)
POWER_COLS = ['15분', '30분', '45분', '60분']
VARIABLES = POWER_COLS + ['평균', '생산량', '기온', '풍속', '습도', '강수량', '전기요금(계절)', '공장인원', '인건비']
sns.set_theme(style='whitegrid', font='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)

def save_table(table, name, index=True):
    table.to_csv(CSV_DIR / f'{name}.csv', index=index, encoding='utf-8-sig')

def date_axis(ax):
    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.set_xlabel('날짜')

raw = pd.read_csv(DATA_PATH)
display(raw.head())
print(f'원본: {raw.shape[0]:,}행 × {raw.shape[1]}열')
print('CSV 저장 위치:', CSV_DIR)

## 1. 데이터 품질과 시간 검증
결측·비정상 시간을 먼저 확인합니다. 날짜별 24개 행과 정상 시간의 행 순서가 일치하는 경우에만 비정상 시간을 행 순서로 복구합니다. 이는 정렬에 근거한 복구 가정이며 실제 시각의 독립적인 검증은 아닙니다.

In [ ]:
quality = pd.DataFrame({'dtype': raw.dtypes.astype(str), '결측수': raw.isna().sum(), '고유값수': raw.nunique()})
display(quality)
print('전체 행 중복:', raw.duplicated().sum())
save_table(quality, 'column_quality')
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
raw.isna().sum().plot.bar(ax=axes[0], color='steelblue')
axes[0].set(title='변수별 결측 개수', ylabel='행 수')
sns.heatmap(raw[VARIABLES].isna().T, cbar=False, ax=axes[1])
axes[1].set(title='결측 위치 (원본 행 순서)', xlabel='행 순서', ylabel='변수')
plt.tight_layout()
plt.show()

df = raw.copy()
df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
df['원본시간'] = df['시간']
order_hour = df.groupby('날짜', sort=False).cumcount()
valid_hour = df['시간'].between(0, 23) & df['시간'].eq(df['시간'].round())
assert df.groupby('날짜').size().eq(24).all(), '날짜별 행 수를 먼저 검토하세요.'
assert df.loc[valid_hour, '시간'].eq(order_hour[valid_hour]).all(), '정상 시간의 정렬을 먼저 검토하세요.'
df['시간수정'] = ~valid_hour
df.loc[~valid_hour, '시간'] = order_hour[~valid_hour]
df['시간'] = df['시간'].astype(int)
df['날짜시간'] = df['날짜'] + pd.to_timedelta(df['시간'], unit='h')
df = df.sort_values('날짜시간').reset_index(drop=True)
assert not df['날짜시간'].duplicated().any()
assert df['날짜시간'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
df['월'] = df['날짜'].dt.month
df['요일번호'] = df['날짜'].dt.dayofweek
df['주말'] = df['요일번호'].ge(5)
df['시간최대전력'] = df[POWER_COLS].max(axis=1)
df['전력평균계산'] = df[POWER_COLS].mean(axis=1)
df['기간'] = np.select([df['월'].isin([1,2]), df['월'].isin([3,4,5]), df['월'].isin([6,7,8])], ['1~2월','3~5월','6~8월'], default='9월')
display(df.loc[df['시간수정'], ['날짜','원본시간','시간','인건비']])
print('수정 시간:', df['시간수정'].sum(), '행 / 기간:', df['날짜시간'].min(), '~', df['날짜시간'].max())
save_table(df.loc[df['시간수정']], 'time_corrections', index=False)

## 2. 파생변수와 반복 패턴
공장인원 계산식이 전력값을 포함하는지 확인합니다. 인건비는 원본과 시간 규칙으로 재생성한 계수를 따로 보존합니다. 반복 패턴은 증강 가능성의 단서이며 자동 삭제하지 않습니다.

In [ ]:
denominator = df[POWER_COLS].sum(axis=1).replace(0, np.nan)
staff_calc = df['생산량'] / denominator
mask = df['공장인원'].notna() & staff_calc.notna()
checks = pd.DataFrame({
    '검사': ['평균과 4개 전력 평균의 최대 차이', '공장인원과 생산량/전력합의 최대 차이', '공장인원 비교 가능 행 수'],
    '값': [(df['평균']-df['전력평균계산']).abs().max(), (df.loc[mask,'공장인원']-staff_calc[mask]).abs().max(), mask.sum()]})
display(checks)
df['인건비_시간규칙'] = np.where(df['시간'].between(9,17), 1.0, 1.5)
print('인건비 원본과 시간 규칙 불일치:', df['인건비'].ne(df['인건비_시간규칙']).sum())
display(df.groupby('월')['전기요금(계절)'].agg(['min','max','nunique']))
calendar_check = pd.DataFrame({'day_요일1시작': df['day'].eq(df['요일번호']+1), 'd_일': df['d'].eq(df['날짜'].dt.day), 'm_월': df['m'].eq(df['월'])}).mean().rename('일치비율')
display(calendar_check)
profiles = df.pivot(index='날짜', columns='시간', values=POWER_COLS)
repeat_counts = pd.Series({'전력4개 조합 반복 행': df.duplicated(POWER_COLS, keep=False).sum(), '일별 전력패턴 반복 날짜': profiles.duplicated(keep=False).sum()})
display(repeat_counts)
fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].scatter(staff_calc[mask], df.loc[mask,'공장인원'], s=8, alpha=.4)
axes[0].set(xlabel='생산량 / 네 전력값 합', ylabel='공장인원', title='공장인원 파생관계')
(df['평균']-df['전력평균계산']).value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(title='평균 컬럼의 반올림 차이', xlabel='원본 평균 − 계산 평균', ylabel='행 수')
plt.tight_layout(); plt.show()
save_table(checks, 'derived_checks', index=False)
save_table(repeat_counts.to_frame('개수'), 'repeated_patterns')

## 3. 15분 시계열 변환
15·30·45·60분을 구간 끝 시각으로 표시한다는 가정입니다. 생산량은 원래 시간 전체의 값이며 네 번 반복됩니다. 합계는 반드시 시간 단위 `df`에서 계산하세요. 결측치는 EDA에서 임의로 채우지 않습니다.

In [ ]:
long_df = df.melt(id_vars=[c for c in df.columns if c not in POWER_COLS], value_vars=POWER_COLS, var_name='분구간', value_name='전력값')
long_df['분'] = long_df['분구간'].str.replace('분','',regex=False).astype(int)
long_df['관측시각'] = long_df['날짜시간'] + pd.to_timedelta(long_df['분'], unit='m')
long_df = long_df.sort_values('관측시각').reset_index(drop=True)
assert len(long_df) == len(df)*4
assert not long_df['관측시각'].duplicated().any()
assert long_df['관측시각'].diff().dropna().eq(pd.Timedelta(minutes=15)).all()
long_df['전력변화량'] = long_df['전력값'].diff()
print('15분 관측:', len(long_df), '개 / 연속성 검사 통과')
display(long_df[['날짜시간','관측시각','분구간','전력값','생산량']].head(8))
# 이후 시간·요일 집계는 원본 시간구간 기준입니다. 60분 값도 원래 구간에 포함됩니다.

## 4. 전체 기간: 변수마다 별도 그래프
전력·생산·기상·비용계수가 같은 시점에 어떻게 바뀌는지 확인합니다.

In [ ]:
for col in VARIABLES:
    fig, ax = plt.subplots(figsize=(15,3))
    ax.plot(df['날짜시간'], df[col], linewidth=.65)
    ax.set(title=f'전체 기간 {col}', ylabel=col)
    date_axis(ax)
    plt.tight_layout(); plt.show()

## 5. 기간별 변화와 월별 전력 분포
기간별 축 범위를 통일해 크기를 비교합니다. 9월은 일부 기간만 존재합니다.

In [ ]:
for col in ['평균','생산량','기온','공장인원','인건비']:
    fig, axes = plt.subplots(2,2, figsize=(16,6), sharey=True)
    for ax, period in zip(axes.flat, ['1~2월','3~5월','6~8월','9월']):
        part = df[df['기간'].eq(period)]
        ax.plot(part['날짜시간'], part[col], linewidth=.65)
        ax.set(title=f'{period} {col}', ylabel=col)
        date_axis(ax)
    plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.boxplot(data=long_df, x='월', y='전력값', showfliers=False, ax=axes[0])
axes[0].set_title('월별 15분 전력 분포 (이상점 표시는 생략)')
df.groupby('월')['생산량'].mean().plot.bar(ax=axes[1], color='steelblue')
axes[1].set(title='월별 시간당 평균 생산량', ylabel='생산량')
plt.tight_layout(); plt.show()

## 6. 시간대·요일·주말 패턴
전체 평균과 평일/주말을 비교하고, 주말 관측만 별도로 표시합니다. 주말 사이의 평일은 선을 연결하지 않습니다.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(15,4))
for weekend, part in df.groupby('주말'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0], marker='o', label='주말' if weekend else '평일')
axes[0].set(title='평일/주말 시간대별 평균전력', ylabel='전력값', xticks=range(0,24,2)); axes[0].legend()
heat = df.pivot_table(index='요일번호', columns='시간', values='전력평균계산', aggfunc='mean')
heat.index = ['월','화','수','목','금','토','일']
sns.heatmap(heat, ax=axes[1], cmap='YlOrRd')
axes[1].set_title('요일 × 시간 평균전력')
plt.tight_layout(); plt.show()
for col in VARIABLES:
    fig, ax = plt.subplots(figsize=(15,2.8))
    ax.plot(df['날짜시간'], df[col].where(df['주말']), linewidth=.8)
    ax.set(title=f'주말 {col}', ylabel=col)
    date_axis(ax)
    plt.tight_layout(); plt.show()

## 7. 생산량과 전력 / 생산량 0인 구간
생산량 0의 전력은 비생산 기저부하 후보입니다. 오류·대기운전 등 다른 가능성을 함께 고려합니다.

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(13,8))
for ax, col in zip(axes.flat, POWER_COLS):
    ax.scatter(df['생산량'], df[col], s=8, alpha=.25)
    ax.set(xlabel='시간별 생산량', ylabel='전력값', title=f'생산량과 {col}')
plt.tight_layout(); plt.show()
df['생산여부'] = np.where(df['생산량'].gt(0), '생산량 > 0', '생산량 = 0')
df['생산량구간'] = pd.qcut(df['생산량'].where(df['생산량'].gt(0)), q=5, duplicates='drop')
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.boxplot(data=df, x='생산여부', y='전력평균계산', ax=axes[0])
sns.boxplot(data=df, x='생산량구간', y='전력평균계산', showfliers=False, ax=axes[1])
axes[0].set_title('생산 여부별 시간 평균전력')
axes[1].set_title('양의 생산량 분위 구간별 전력'); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(12,4))
for label, part in df.groupby('생산여부'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=ax, marker='o', label=label)
ax.set(title='생산 여부별 시간대 전력', ylabel='전력값'); ax.legend()
plt.tight_layout(); plt.show()

## 8. 기상조건과 전력
관계가 보여도 생산량·시간·계절이 함께 바뀌므로 인과관계로 단정하지 않습니다.

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(13,8))
for ax, col in zip(axes.flat, ['기온','습도','풍속','강수량']):
    sns.scatterplot(data=df, x=col, y='전력평균계산', hue='생산여부', s=12, alpha=.3, ax=ax)
    ax.set_title(f'{col}과 전력')
plt.tight_layout(); plt.show()
cols = ['전력평균계산','생산량','기온','습도','풍속','강수량']
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[cols].corr(method='spearman'), annot=True, fmt='.2f', vmin=-1, vmax=1, cmap='coolwarm', ax=ax)
ax.set_title('Spearman 상관계수'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(data=df, x='생산량구간', y='전력평균계산', hue='주말', showfliers=False, ax=ax)
ax.set_title('비슷한 생산량 구간에서 평일/주말 전력 비교')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 9. 급상승과 고전력 후보
전력값 상위 10%·5%·1%의 경계값을 비교합니다. 같은 값이 많으면 경계값 이상 비율이 명목 비율보다 클 수 있습니다. 아래 기준은 전체 기간 EDA용이며 계약전력 기준이 아닙니다.

In [ ]:
thresholds = long_df['전력값'].quantile([.90,.95,.99])
peak_summary = pd.DataFrame({'기준값': thresholds, '기준이상비율': [long_df['전력값'].ge(v).mean() for v in thresholds]})
display(peak_summary)
fig, axes = plt.subplots(1,2, figsize=(15,4))
sns.histplot(long_df['전력값'], bins=60, ax=axes[0])
for q, value in thresholds.items():
    axes[0].axvline(value, linestyle='--', label=f'{q:.0%}: {value:g}')
    long_df.assign(고전력=long_df['전력값'].ge(value)).groupby('시간')['고전력'].mean().mul(100).plot(ax=axes[1], marker='o', label=f'{q:.0%} 기준')
axes[0].legend(); axes[0].set_title('전력 분포와 후보 기준')
axes[1].set(title='시간대별 고전력 발생률', ylabel='발생률 (%)'); axes[1].legend()
plt.tight_layout(); plt.show()
long_df['피크후보95'] = long_df['전력값'].ge(thresholds.loc[.95])
fig, axes = plt.subplots(2,1, figsize=(15,7))
axes[0].plot(long_df['관측시각'], long_df['전력값'], linewidth=.5)
p = long_df[long_df['피크후보95']]
axes[0].scatter(p['관측시각'], p['전력값'], color='tomato', s=7, label='95% 기준 이상')
axes[0].set(title='고전력 후보 시점', ylabel='전력값'); axes[0].legend()
axes[1].plot(long_df['관측시각'], long_df['전력변화량'], linewidth=.5)
axes[1].set(title='직전 15분 대비 전력 변화', ylabel='전력값 차이')
for ax in axes: date_axis(ax)
plt.tight_layout(); plt.show()
display(long_df.nlargest(15, '전력변화량')[['관측시각','전력값','전력변화량','생산량','시간수정']])
peak_by_hour = long_df.groupby('시간').agg(관측수=('전력값','size'), 피크수=('피크후보95','sum'), 발생률=('피크후보95','mean'))
display(peak_by_hour)
save_table(peak_summary, 'peak_threshold_candidates')
save_table(peak_by_hour, 'peak_by_hour')

## 10. CSV 저장 및 다음 단계
분석용 시간별/15분별 데이터와 요약표를 저장합니다. 그래프 파일은 저장하지 않습니다.

다음 모델 단계에서는 과거 전력·시간정보를 기본으로 하고, 생산계획 및 기상의 사전 이용 가능성을 확인한 뒤 추가합니다. 같은 시간의 평균과 공장인원은 제외합니다. 시간순 분할 후 반복 패턴의 평가 영향을 별도로 점검합니다.

In [ ]:
monthly_summary = df.groupby('월').agg(관측수=('평균','size'), 평균전력=('전력평균계산','mean'), 최대전력=('시간최대전력','max'), 평균생산량=('생산량','mean'))
save_table(monthly_summary, 'monthly_summary')
save_table(df, 'hourly_eda', index=False)
save_table(long_df, 'quarter_hour_eda', index=False)
save_table(df[VARIABLES].describe().T, 'descriptive_statistics')
display(monthly_summary)
print('저장된 CSV:')
for path in sorted(CSV_DIR.glob('*.csv')):
    print(path.name)

## 11. 생산량 0인데 전력이 높은 구간
생산량 0인 시간의 평균전력 상위 10%를 탐색 기준으로 사용합니다. 앞 절의 15분 전력 95% 기준과는 다른 기준입니다. 연속 구간의 길이와 생산 전후 여부를 확인하며, 원인을 대기부하나 오류로 단정하지 않습니다.


In [ ]:
zero_mask = df['생산량'].eq(0)
zero_threshold = df.loc[zero_mask, '전력평균계산'].quantile(.9)
zero_high = zero_mask & df['전력평균계산'].ge(zero_threshold)
zero_analysis = df[['날짜시간','시간','월','요일번호','생산량','전력평균계산','시간수정']].copy()
zero_analysis['고전력_비생산'] = zero_high
zero_analysis['직전시간_생산중'] = df['생산량'].shift(1).gt(0)
zero_analysis['다음시간_생산중'] = df['생산량'].shift(-1).gt(0)
zero_run_id = zero_mask.ne(zero_mask.shift()).cumsum()
zero_runs = df.loc[zero_mask].groupby(zero_run_id[zero_mask]).agg(
    시작=('날짜시간','min'), 마지막관측=('날짜시간','max'),
    지속시간=('날짜시간','size'), 평균전력=('전력평균계산','mean'), 최대전력=('시간최대전력','max'))
zero_runs['종료'] = zero_runs['마지막관측'] + pd.Timedelta(hours=1)
print(f'비생산 시간 평균전력 90% 기준: {zero_threshold:.2f}; 해당 시간: {zero_high.sum()}개')
fig, axes = plt.subplots(2,2, figsize=(15,8))
sns.histplot(df.loc[zero_mask,'전력평균계산'], bins=40, ax=axes[0,0])
axes[0,0].axvline(zero_threshold, color='tomato', linestyle='--')
axes[0,0].set_title('생산량 0: 평균전력 분포와 상위 10% 경계')
axes[0,1].plot(df['날짜시간'], df['전력평균계산'].where(zero_mask), linewidth=.6)
axes[0,1].scatter(df.loc[zero_high,'날짜시간'], df.loc[zero_high,'전력평균계산'], color='tomato', s=12)
axes[0,1].set_title('비생산 전력과 높은 전력 시점'); date_axis(axes[0,1])
zero_hour = zero_analysis.loc[zero_mask].groupby('시간').agg(비생산관측수=('고전력_비생산','size'), 높은전력수=('고전력_비생산','sum'), 발생률=('고전력_비생산','mean'))
zero_hour['발생률'].mul(100).plot.bar(ax=axes[1,0])
axes[1,0].set(title='비생산 시간 중 높은 전력 발생률', ylabel='%')
sns.histplot(zero_runs['지속시간'], bins=25, ax=axes[1,1])
axes[1,1].set(title='연속 비생산 구간의 길이', xlabel='시간')
plt.tight_layout(); plt.show()
display(zero_hour)
display(zero_analysis.loc[zero_high].sort_values('전력평균계산', ascending=False).head(20))
save_table(zero_analysis.loc[zero_high], 'zero_production_high_power', index=False)
save_table(zero_runs, 'zero_production_runs')
save_table(zero_hour, 'zero_production_by_hour')


## 12. 생산 시작·종료 전후 변화
0→양수를 시작, 양수→0을 종료로 정의합니다. 전후 3시간이 모두 존재하는 사건만 비교하며 평균과 25~75% 범위를 표시합니다. 가까운 사건의 창이 겹칠 수 있으며, 관측된 전환이 실제 설비 가동 시점과 같다는 보장은 없습니다.


In [ ]:
def event_windows(frame, positions, offsets, columns):
    records = []
    for event_id, pos in enumerate(positions):
        if pos + min(offsets) < 0 or pos + max(offsets) >= len(frame):
            continue
        for offset in offsets:
            row = frame.iloc[pos + offset]
            records.append({'사건번호': event_id, '상대시간': offset, **{col: row[col] for col in columns}})
    return pd.DataFrame(records, columns=['사건번호','상대시간'] + columns)

def plot_event_profile(windows, column, ax, title, xlabel):
    if windows.empty:
        ax.set_title(title + ' (완전한 관측창 없음)')
        return
    grouped = windows.groupby('상대시간')[column]
    stats = grouped.agg(['mean','count'])
    lower, upper = grouped.quantile(.25), grouped.quantile(.75)
    ax.plot(stats.index, stats['mean'], marker='o', label='평균')
    ax.fill_between(stats.index, lower, upper, alpha=.2, label='25~75%')
    ax.axvline(0, color='tomato', linestyle='--')
    ax.set(title=f"{title} (사건 {windows['사건번호'].nunique()}개)", xlabel=xlabel, ylabel=column)
    ax.legend()

previous_production = df['생산량'].shift(1)
start_positions = np.flatnonzero(previous_production.eq(0) & df['생산량'].gt(0))
end_positions = np.flatnonzero(previous_production.gt(0) & df['생산량'].eq(0))
transition_windows = {}
fig, axes = plt.subplots(2,2, figsize=(14,8))
for row, (label, positions) in enumerate([('생산 시작',start_positions), ('생산 종료',end_positions)]):
    windows = event_windows(df, positions, range(-3,4), ['전력평균계산','생산량'])
    transition_windows[label] = windows
    for ax, col in zip(axes[row], ['전력평균계산','생산량']):
        plot_event_profile(windows, col, ax, label, '전환 기준 상대시간 (시간)')
    save_table(windows, 'production_start_windows' if row == 0 else 'production_end_windows', index=False)
plt.tight_layout(); plt.show()
transition_counts = pd.DataFrame({'시작':df.iloc[start_positions]['시간'].value_counts(), '종료':df.iloc[end_positions]['시간'].value_counts()}).reindex(range(24)).fillna(0).astype(int)
transition_counts.plot.bar(figsize=(13,4))
plt.title('시간대별 생산 시작·종료 횟수'); plt.ylabel('사건 수')
plt.tight_layout(); plt.show()
save_table(transition_counts, 'production_transition_counts')


## 13. 피크 시작 전후 패턴과 지속시간
15분 전력의 95% 경계 이상이 이어지는 구간을 하나의 사건으로 묶습니다. 개별 고전력 점을 모두 사건으로 세지 않고 각 구간의 첫 관측을 기준으로 전후 2시간을 비교합니다. 데이터 양 끝에 걸친 사건은 실제 지속시간이 더 길 수 있습니다. 생산량은 시간 단위 값이 반복된 참고값입니다.


In [ ]:
peak_mask = long_df['피크후보95']
peak_run_id = peak_mask.ne(peak_mask.shift()).cumsum()
peak_events = long_df.loc[peak_mask].groupby(peak_run_id[peak_mask]).agg(
    첫관측=('관측시각','min'), 마지막관측=('관측시각','max'),
    관측수=('전력값','size'), 최대전력=('전력값','max'))
peak_events['지속분'] = peak_events['관측수'] * 15
peak_events['경계사건'] = peak_events['첫관측'].eq(long_df['관측시각'].min()) | peak_events['마지막관측'].eq(long_df['관측시각'].max())
peak_starts = np.flatnonzero(peak_mask & ~peak_mask.shift(1, fill_value=False))
peak_windows = event_windows(long_df, peak_starts, range(-8,9), ['전력값','생산량'])
peak_windows['상대시간'] = peak_windows['상대시간'] * 15
fig, axes = plt.subplots(2,2, figsize=(14,8))
plot_event_profile(peak_windows, '전력값', axes[0,0], '고전력 구간 시작 전후', '첫 고전력 관측 기준 (분)')
plot_event_profile(peak_windows, '생산량', axes[0,1], '원본 시간별 생산량', '첫 고전력 관측 기준 (분)')
duration_group = pd.cut(peak_events['지속분'], [0,15,30,45,np.inf], labels=['15분','30분','45분','60분 이상'])
duration_group.value_counts(sort=False).plot.bar(ax=axes[1,0])
axes[1,0].set(title='고전력 사건 지속시간', ylabel='사건 수')
long_df.iloc[peak_starts]['시간'].value_counts().reindex(range(24), fill_value=0).plot.bar(ax=axes[1,1])
axes[1,1].set(title='원본 시간구간별 고전력 시작 횟수', xlabel='원본 시간', ylabel='사건 수')
plt.tight_layout(); plt.show()
display(peak_events.sort_values('지속분', ascending=False).head(15))
save_table(peak_events, 'peak_events')
save_table(peak_windows, 'peak_event_windows', index=False)


## 14. 비슷한 생산량에서 시간대별 전력 비교
양의 생산량을 5개 분위 구간으로 나누고 시간대별 중앙값·90백분위수를 비교합니다. 표본 10개 미만인 조합은 전력 지도에서 가립니다. 같은 구간에서도 정확한 생산량·기상·요일은 다르므로 시간 이동의 인과효과를 뜻하지 않습니다.


In [ ]:
positive = df.loc[df['생산량'].gt(0)].copy()
matched_summary = positive.groupby(['생산량구간','시간'], observed=True).agg(
    관측수=('전력평균계산','size'), 중앙값=('전력평균계산','median'),
    상위90=('전력평균계산', lambda x: x.quantile(.9)),
    평균생산량=('생산량','mean')).reset_index()
count_grid = matched_summary.pivot(index='생산량구간', columns='시간', values='관측수').reindex(columns=range(24)).fillna(0)
fig, axes = plt.subplots(3,1, figsize=(16,11))
for ax, metric in zip(axes[:2], ['중앙값','상위90']):
    grid = matched_summary.pivot(index='생산량구간', columns='시간', values=metric).reindex(columns=range(24))
    sns.heatmap(grid, mask=count_grid.lt(10), cmap='YlOrRd', annot=True, fmt='.0f', ax=ax)
    ax.set_title(f'생산량 구간 × 시간: 전력 {metric} (n≥10)')
sns.heatmap(count_grid, cmap='Blues', annot=True, fmt='.0f', ax=axes[2])
axes[2].set_title('조합별 표본 수')
plt.tight_layout(); plt.show()
display(matched_summary.sort_values(['생산량구간','시간']))
save_table(matched_summary, 'production_bin_hour_power', index=False)
print('추가 분석 CSV 저장 위치:', CSV_DIR)


## 15. 이상 날짜·0전력·반복 패턴 민감도
전체 자료의 15분 전력 95% 기준을 고정해 비교합니다. 제외 후 기준을 다시 계산하면 서로 다른 피크를 비교하게 됩니다. 0전력은 자동으로 오류라고 판단하지 않습니다. 중복 일별 패턴의 첫 날짜만 남기는 결과는 날짜 선택에 민감한 보조 분석입니다.


In [ ]:
daily_profiles = df.pivot(index='날짜', columns='시간', values=POWER_COLS).sort_index()
duplicate_days = daily_profiles.index[daily_profiles.duplicated(keep=False)]
representative_days = daily_profiles.index[~daily_profiles.duplicated(keep='first')]
suspect_dates = pd.to_datetime(['2021-07-13','2021-07-15'])
sensitivity_sets = {
    '전체': df,
    '이상 날짜 제외': df.loc[~df['날짜'].isin(suspect_dates)],
    '네 전력 모두 0 제외': df.loc[~df[POWER_COLS].eq(0).all(axis=1)],
    '동일 일별패턴 첫 날짜만': df.loc[df['날짜'].isin(representative_days)]
}
fixed_threshold = long_df['전력값'].quantile(.95)
sensitivity_rows = []
fig, axes = plt.subplots(1,2, figsize=(15,5))
monthly_sensitivity = []
for label, part in sensitivity_sets.items():
    flag = part[POWER_COLS].ge(fixed_threshold)
    sensitivity_rows.append({'조건':label, '시간수':len(part), '날짜수':part['날짜'].nunique(),
        '평균전력':part['전력평균계산'].mean(), '피크비율':flag.to_numpy().mean(),
        '평균생산량':part['생산량'].mean(), '주말비율':part['주말'].mean()})
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0], label=label)
    monthly = flag.mean(axis=1).groupby(part['월']).mean()
    monthly.mul(100).plot(ax=axes[1], marker='o', label=label)
    monthly_sensitivity.append(monthly.rename(label))
axes[0].set(title='제외 조건별 시간대 평균전력', ylabel='전력값')
axes[1].set(title=f'동일 기준 {fixed_threshold:g} 이상: 월별 비율', ylabel='%')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
sensitivity_summary = pd.DataFrame(sensitivity_rows).set_index('조건')
display(sensitivity_summary)
save_table(sensitivity_summary, 'sensitivity_summary')
save_table(pd.concat(monthly_sensitivity, axis=1), 'sensitivity_monthly_peak')


## 16. 반복 일별 패턴과 고유 패턴 비교
24시간×4개 전력값이 모두 같은 날짜를 같은 프로파일로 분류합니다. 반복 여부는 전체 기간을 이용한 사후 EDA 정보이며 예측 입력으로 사용하지 않습니다. 월·요일 구성 차이가 전력 차이에 영향을 줄 수 있어 구성도 함께 표시합니다.


In [ ]:
repeat_eda = df.copy()
repeat_eda['패턴집단'] = np.where(repeat_eda['날짜'].isin(duplicate_days), '반복 패턴 날짜', '고유 패턴 날짜')
repeat_eda['피크비율'] = repeat_eda[POWER_COLS].ge(fixed_threshold).mean(axis=1)
repeat_summary = repeat_eda.groupby('패턴집단').agg(날짜수=('날짜','nunique'), 시간수=('날짜','size'),
    평균전력=('전력평균계산','mean'), 평균생산량=('생산량','mean'), 피크비율=('피크비율','mean'))
display(repeat_summary)
fig, axes = plt.subplots(2,2, figsize=(15,9))
for label, part in repeat_eda.groupby('패턴집단'):
    part.groupby('시간')['전력평균계산'].mean().plot(ax=axes[0,0], label=label)
    part.groupby('시간')['피크비율'].mean().mul(100).plot(ax=axes[0,1], label=label)
axes[0,0].set(title='패턴 집단별 시간대 평균전력', ylabel='전력값')
axes[0,1].set(title='패턴 집단별 고전력 비율', ylabel='%')
for ax in axes[0]: ax.legend()
daily_groups = repeat_eda.drop_duplicates('날짜')
pd.crosstab(daily_groups['월'], daily_groups['패턴집단']).plot.bar(ax=axes[1,0])
axes[1,0].set(title='집단별 월 구성', ylabel='날짜 수')
pd.crosstab(daily_groups['요일번호'], daily_groups['패턴집단']).plot.bar(ax=axes[1,1])
axes[1,1].set(title='집단별 요일 구성 (0=월, 6=일)', ylabel='날짜 수')
plt.tight_layout(); plt.show()
# 동일 프로파일 날짜 목록: 해시 대신 실제 전력 열 전체로 그룹화
profile_ids = daily_profiles.groupby(list(daily_profiles.columns), sort=False).ngroup().rename('프로파일ID')
profile_catalog = profile_ids.reset_index()
profile_catalog['같은프로파일날짜수'] = profile_catalog.groupby('프로파일ID')['날짜'].transform('size')
display(profile_catalog.loc[profile_catalog['같은프로파일날짜수'].gt(1)].head(20))
save_table(repeat_summary, 'repeated_vs_unique_summary')
save_table(profile_catalog, 'daily_profile_catalog', index=False)


## 16-1. 같은 전력 프로파일 안의 생산량·기상 비교
반복 그룹마다 날짜 목록·연속 출현일 사이 간격·요일 구성과 변수별 24시간 값의 일치 여부를 확인합니다. 같은 위치의 결측은 패턴 비교에서 같게 취급하되 결측 수를 별도로 표시합니다. 수치가 조금이라도 다르면 서로 다른 패턴으로 셉니다.

아래 선택 셀의 `SELECT_PROFILE_ID`를 바꾸면 다른 그룹을 볼 수 있습니다. 기본값 None은 가장 많이 반복된 그룹을 선택합니다. 날짜 간격은 나열된 인접 날짜 사이의 간격이며, 생성 의도나 증강 규칙을 확정하는 근거는 아닙니다.


In [ ]:
# 16번에서 만든 profile_catalog와 원본 시간별 df를 연결합니다.
repeat_detail = df.merge(
    profile_catalog[['날짜','프로파일ID','같은프로파일날짜수']],
    on='날짜', how='left', validate='many_to_one'
)
repeat_detail = repeat_detail.loc[repeat_detail['같은프로파일날짜수'].gt(1)].copy()
COMPARE_COLS = ['생산량','기온','풍속','습도','강수량','전기요금(계절)','공장인원','인건비']
weekday_names = ['월','화','수','목','금','토','일']
group_rows, match_rows, gap_rows = [], [], []
for profile_id, part in repeat_detail.groupby('프로파일ID'):
    dates = pd.DatetimeIndex(part['날짜'].unique()).sort_values()
    gaps = dates.to_series().diff().dt.days.dropna().astype(int)
    for previous, current in zip(dates[:-1], dates[1:]):
        gap_rows.append({'프로파일ID':profile_id, '이전날짜':previous,
                         '다음날짜':current, '간격일':(current-previous).days})
    group_rows.append({
        '프로파일ID':profile_id, '반복날짜수':len(dates),
        '날짜목록':', '.join(dates.strftime('%Y-%m-%d')),
        '요일목록':', '.join(weekday_names[i] for i in sorted(set(dates.dayofweek))),
        '최소간격일':gaps.min(), '중앙간격일':gaps.median(), '최대간격일':gaps.max(),
        '7일간격비율':gaps.eq(7).mean(),
        '7일배수간격비율':gaps.mod(7).eq(0).mean()
    })
    # 96개 전력값 일치 여부를 독립적으로 확인합니다.
    power_matrix = part.pivot(index='날짜', columns='시간', values=POWER_COLS)
    assert len(power_matrix.drop_duplicates()) == 1
    for col in COMPARE_COLS:
        matrix = part.pivot(index='날짜', columns='시간', values=col).reindex(columns=range(24))
        unique_patterns = len(matrix.drop_duplicates())
        match_rows.append({'프로파일ID':profile_id, '변수':col,
            '반복날짜수':len(dates), '고유24시간패턴수':unique_patterns,
            '모든날짜동일':unique_patterns==1, '결측수':int(matrix.isna().sum().sum()),
            '날짜간최대시간별차이':(matrix.max(axis=0)-matrix.min(axis=0)).max()})
repeat_group_summary = pd.DataFrame(group_rows).sort_values(
    ['반복날짜수','프로파일ID'], ascending=[False,True])
repeat_variable_matches = pd.DataFrame(match_rows)
repeat_date_gaps = pd.DataFrame(gap_rows)
display(repeat_group_summary)
match_matrix = repeat_variable_matches.pivot(index='프로파일ID', columns='변수', values='모든날짜동일')
display(match_matrix.reindex(repeat_group_summary['프로파일ID']).rename_axis(columns='변수: True=24시간 값 모두 동일'))
fig, ax = plt.subplots(figsize=(12, max(5, len(match_matrix)*.22)))
sns.heatmap(match_matrix.reindex(repeat_group_summary['프로파일ID'])[COMPARE_COLS].astype(int),
            vmin=0, vmax=1, cmap=['#eeeeee','#287da8'], cbar=False, ax=ax)
ax.set(title='동일 전력 그룹 안의 변수 일치: 파랑=동일, 회색=다름', ylabel='프로파일ID')
plt.tight_layout(); plt.show()
save_table(repeat_group_summary, 'repeat_group_summary', index=False)
save_table(repeat_variable_matches, 'repeat_variable_matches', index=False)
save_table(repeat_date_gaps, 'repeat_date_gaps', index=False)
save_table(repeat_detail, 'repeat_group_hourly_data', index=False)


In [ ]:
# 다른 그룹을 보려면 None 대신 위 표의 프로파일ID를 입력하세요. 예: 4
SELECT_PROFILE_ID = None
selected_profile = int(repeat_group_summary.iloc[0]['프로파일ID']) if SELECT_PROFILE_ID is None else SELECT_PROFILE_ID
selected = repeat_detail.loc[repeat_detail['프로파일ID'].eq(selected_profile)].sort_values(['날짜','시간'])
assert not selected.empty, '반복그룹 목록에 있는 프로파일ID를 선택하세요.'
selected_dates = sorted(selected['날짜'].unique())
daily_comparison = selected.groupby('날짜').agg(
    요일번호=('요일번호','first'), 총생산량=('생산량',lambda s:s.sum(min_count=1)),
    평균기온=('기온','mean'), 평균습도=('습도','mean'), 평균풍속=('풍속','mean'),
    평균강수량관측값=('강수량','mean'), 계절계수=('전기요금(계절)','first'),
    시간수정수=('시간수정','sum'))
daily_comparison['요일'] = daily_comparison.pop('요일번호').map(dict(enumerate(weekday_names)))
print(f'선택 그룹: {selected_profile} / {len(selected_dates)}일')
display(daily_comparison)
display(repeat_variable_matches.loc[repeat_variable_matches['프로파일ID'].eq(selected_profile)])
# 전력 96개는 완전히 같으므로 대표 날짜 하나만 표시합니다.
representative = selected.loc[selected['날짜'].eq(selected_dates[0])].sort_values('시간')
fig, ax = plt.subplots(figsize=(14,3.5))
ax.plot(np.arange(1,97)*.25, representative[POWER_COLS].to_numpy().ravel())
ax.set(title=f'그룹 {selected_profile}: 공통 전력 곡선 ({len(selected_dates)}일 동일)',
       xlabel='하루 시작 후 시간 (구간 끝)', ylabel='전력값', xticks=range(0,25,2))
plt.tight_layout(); plt.show()
# 변수별로 날짜를 색으로 구분합니다. 값이 같으면 선이 겹칩니다.
colors = plt.cm.turbo(np.linspace(.05,.95,len(selected_dates)))
for col in COMPARE_COLS:
    fig, ax = plt.subplots(figsize=(14,4))
    for date, color in zip(selected_dates, colors):
        part = selected.loc[selected['날짜'].eq(date)]
        ax.plot(part['시간'],part[col],color=color,lw=1,alpha=.8,
                label=pd.Timestamp(date).strftime('%Y-%m-%d'))
    ax.set(title=f'그룹 {selected_profile}: 날짜별 {col}',xlabel='시간',ylabel=col,xticks=range(0,24,2))
    ax.legend(loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,ncol=max(1,(len(selected_dates)+19)//20))
    plt.tight_layout(); plt.show()
save_table(daily_comparison, f'repeat_profile_{selected_profile}_daily_summary')
print('생산량과 기상이 달라도 전력은 동일할 수 있습니다. 일치 결과만으로 데이터 제작 의도를 확정하지 않습니다.')


## 17. 한 시간 안의 전력 상승과 변동
60분−15분은 시간 내 변화, 최대−최소는 시간 내 범위입니다. 해당 시간이 끝나야 알 수 있는 값이므로 같은 시간 미래 전력의 입력으로 쓰면 안 됩니다. 다음 시간 예측에는 이용 시점을 확인한 뒤 과거 정보로 사용할 수 있습니다.


In [ ]:
within_hour = df[['날짜시간','시간','생산량','생산여부']].copy()
within_hour['시간내변화'] = df['60분'] - df['15분']
within_hour['시간내범위'] = df[POWER_COLS].max(axis=1) - df[POWER_COLS].min(axis=1)
quarter_summary = pd.DataFrame({'평균':df[POWER_COLS].mean(), '중앙값':df[POWER_COLS].median(),
    '피크비율':df[POWER_COLS].ge(fixed_threshold).mean()})
fig, axes = plt.subplots(2,2, figsize=(14,8))
quarter_summary[['평균','중앙값']].plot.bar(ax=axes[0,0])
axes[0,0].set_title('15·30·45·60분 전력 수준')
quarter_summary['피크비율'].mul(100).plot.bar(ax=axes[0,1])
axes[0,1].set(title='분 구간별 고전력 비율', ylabel='%')
sns.boxplot(data=within_hour, x='시간', y='시간내변화', showfliers=False, ax=axes[1,0])
axes[1,0].axhline(0, color='gray', linestyle='--')
axes[1,0].set_title('시간대별 60분−15분 (이상점 표시 생략)')
axes[1,1].scatter(within_hour['생산량'], within_hour['시간내범위'], s=9, alpha=.25)
axes[1,1].set(title='생산량과 시간 내 변동 범위', xlabel='생산량', ylabel='최대−최소 전력')
plt.tight_layout(); plt.show()
display(quarter_summary)
save_table(quarter_summary, 'quarter_position_summary')
save_table(within_hour, 'within_hour_variation', index=False)


## 18. 생산전력비율 탐색
생산량/네 전력값 합을 과거 관측의 설명 지표로만 사용합니다. 실제 인원·에너지 효율·비용이 아닙니다. 생산량 양수 및 분모 양수인 시간만 비교합니다. 낮은 비율이 곧 낭비라는 뜻은 아니며 제품·공정 차이를 확인할 데이터가 없습니다.


In [ ]:
ratio_eda = df.loc[df['생산량'].gt(0) & df[POWER_COLS].sum(axis=1).gt(0)].copy()
ratio_eda['생산전력비율'] = ratio_eda['생산량'] / ratio_eda[POWER_COLS].sum(axis=1)
fig, axes = plt.subplots(1,3, figsize=(18,5))
sns.boxplot(data=ratio_eda, x='시간', y='생산전력비율', showfliers=False, ax=axes[0])
sns.boxplot(data=ratio_eda, x='월', y='생산전력비율', showfliers=False, ax=axes[1])
sns.boxplot(data=ratio_eda, x='생산량구간', y='생산전력비율', showfliers=False, ax=axes[2])
for ax,title in zip(axes, ['시간대별 비율','월별 비율','생산량 구간별 비율']):
    ax.set_title(title + ' (이상점 표시 생략)')
axes[2].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()
ratio_summary = ratio_eda.groupby('시간')['생산전력비율'].agg(['count','median','mean'])
display(ratio_summary)
save_table(ratio_summary, 'production_power_ratio_by_hour')
save_table(ratio_eda[['날짜시간','생산량','생산전력비율']], 'production_power_ratio', index=False)


## 19. 전력 시차 관계와 생산량의 선행·후행 관계
관측 간 상관을 보는 EDA입니다. 예측 성능 평가와 인과 검정은 아닙니다. 모든 전력 시차는 같은 관측 기간으로 비교합니다. 양의 생산량 시차는 과거 생산량, 음수는 미래 생산량으로 미래 값은 진단에만 사용합니다. 증강 반복 패턴이 상관을 높일 수 있습니다.


In [ ]:
# ============================================================
# 1. 전력 시차 상관관계
# ============================================================

# long_df는 관측시각 기준으로 정렬되어 있어야 함
long_df = long_df.sort_values('관측시각').reset_index(drop=True)

# 15분 간격의 고정 시차
lag_steps = {
    '15분 전': 1,
    '30분 전': 2,
    '1시간 전': 4,
    '24시간 전': 96,
    '1주 전': 672,
    '2주 전': 1344,
    '3주 전': 2016,
    '4주 전': 2688
}

lag_frame = pd.DataFrame({
    '관측시각': long_df['관측시각'],
    '현재전력': long_df['전력값']
})

# 고정된 15분 단위 시차 생성
for label, steps in lag_steps.items():
    lag_frame[label] = long_df['전력값'].shift(steps)


# ------------------------------------------------------------
# 달력 기준 한 달 전 같은 시각의 전력
# ------------------------------------------------------------

# 관측시각별 전력 조회용 Series
power_by_time = long_df.set_index('관측시각')['전력값']

# 각 현재 시각에서 달력 기준으로 한 달 전 시각 계산
previous_month_time = (
    long_df['관측시각']
    .apply(lambda x: x - pd.DateOffset(months=1))
)

# 한 달 전 같은 시각의 전력 연결
lag_frame['한 달 전'] = power_by_time.reindex(
    previous_month_time
).to_numpy()


# ------------------------------------------------------------
# 모든 시차를 동일한 관측 구간에서 비교
# ------------------------------------------------------------

comparison_cols = list(lag_steps.keys()) + ['한 달 전']

lag_common = lag_frame[
    ['현재전력'] + comparison_cols
].dropna()

lag_summary = pd.DataFrame({
    'Pearson': [
        lag_common['현재전력'].corr(lag_common[col])
        for col in comparison_cols
    ],
    'Spearman': [
        lag_common['현재전력'].corr(
            lag_common[col],
            method='spearman'
        )
        for col in comparison_cols
    ],
    '관측수': len(lag_common)
}, index=comparison_cols)


# ------------------------------------------------------------
# 시차별 산점도
# ------------------------------------------------------------

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flat

for ax, label in zip(axes, comparison_cols):
    ax.hexbin(
        lag_common[label],
        lag_common['현재전력'],
        gridsize=35,
        mincnt=1,
        cmap='Blues'
    )

    ax.set(
        title=(
            f"{label}: "
            f"Pearson={lag_summary.loc[label, 'Pearson']:.3f}"
        ),
        xlabel=label,
        ylabel='현재전력'
    )

# 사용하지 않은 축만 숨김
for ax in list(axes)[len(comparison_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

display(lag_summary)


# ============================================================
# 2. 생산량 시차와 전력의 상관관계
# ============================================================

# 비교 시각을 통일하고 양끝 6시간 제외
production_lags = pd.DataFrame({
    lag: df['생산량'].shift(lag)
    for lag in range(-6, 7)
})

production_lags['전력'] = df['전력평균계산']
production_lags = production_lags.dropna()

cross_corr = pd.Series({
    lag: production_lags[lag].corr(
        production_lags['전력'],
        method='spearman'
    )
    for lag in range(-6, 7)
}, name='Spearman')

fig, ax = plt.subplots(figsize=(11, 4))

cross_corr.plot.bar(ax=ax)

ax.set(
    title='생산량 시차와 현재 전력의 관계',
    xlabel='생산량 시차 (시간): 양수=과거, 음수=미래',
    ylabel='상관계수'
)

plt.tight_layout()
plt.show()

display(cross_corr.to_frame())


# ============================================================
# 3. 결과 저장
# ============================================================

save_table(
    lag_summary,
    'power_lag_correlations'
)

save_table(
    cross_corr.to_frame(),
    'production_power_cross_correlations'
)

## 20. EDA 결과 정리
다음 질문의 답을 그래프와 저장된 표를 근거로 정리하세요.

- 이상 날짜를 제외해도 08시·7월 피크 집중이 유지되는가?
- 반복/고유 패턴 집단의 차이에 월·요일 구성 차이가 섞여 있는가?
- 생산 시작·종료와 고전력 사건의 전후 변화가 일관적인가?
- 같은 생산량 구간의 시간대 비교에 충분한 표본이 있는가?
- 과거 전력 중 어떤 시차가 강한 관계를 보이는가?

다음 단계는 시간순 학습·검증·테스트 분할과 기준모델입니다. 이 노트북에서 전체 기간으로 계산한 피크 경계·분위 구간을 모델 평가에 그대로 재사용하지 않습니다. 그래프는 저장하지 않으며 추가 표는 기존 `results/eda/csv` 폴더에 저장됩니다.
